In [2]:
import json
import pandas as pd

# Load the data we downloaded
with open('dataset/all_samples.json', 'r') as f:
    data = json.load(f)

print(f"Total samples: {len(data)}")
print(f"\nFirst sample:")
print("=" * 50)
print(data[0]['code'])

Total samples: 10000

First sample:
def __msgc_step3_discontinuity_localization(self):
        """
        Estimate discontinuity in basis of low resolution image segmentation.
        :return: discontinuity in low resolution
        """
        import scipy

        start = self._start_time
        seg = 1 - self.segmentation.astype(np.int8)
        self.stats["low level object voxels"] = np.sum(seg)
        self.stats["low level image voxels"] = np.prod(seg.shape)
        # in seg is now stored low resolution segmentation
        # back to normal parameters
        # step 2: discontinuity localization
        # self.segparams = sparams_hi
        seg_border = scipy.ndimage.filters.laplace(seg, mode="constant")
        logger.debug("seg_border: %s", scipy.stats.describe(seg_border, axis=None))
        # logger.debug(str(np.max(seg_border)))
        # logger.debug(str(np.min(seg_border)))
        seg_border[seg_border != 0] = 1
        logger.debug("seg_border: %s", scipy.stats.describ

In [3]:
# Let's look at 5 random samples to get variety
import random

random.seed(42)  # So we see the same ones
sample_indices = random.sample(range(len(data)), 5)

for i, idx in enumerate(sample_indices):
    print(f"\n{'='*60}")
    print(f"Sample {i+1} - Function: {data[idx]['name']}")
    print(f"{'='*60}")
    print(data[idx]['code'][:500])  # First 500 characters
    print("..." if len(data[idx]['code']) > 500 else "")


Sample 1 - Function: temp_dir
def temp_dir(folder=None, delete=True):
    # type: (Optional[str], bool) -> str
    """Get a temporary directory optionally with folder appended (and created if it doesn't exist)

    Args:
        folder (Optional[str]): Folder to create in temporary folder. Defaults to None.
        delete (bool): Whether to delete folder on exiting with statement

    Returns:
        str: A temporary directory
    """
    tempdir = get_temp_dir()
    if folder:
        tempdir = join(tempdir, folder)
    if
...

Sample 2 - Function: doublewrap
def doublewrap(f):
    '''
    a decorator decorator, allowing the decorator to be used as:
    @decorator(with, arguments, and=kwargs)
    or
    @decorator
    Ref: http://stackoverflow.com/questions/653368/how-to-create-a-python-decorator-that-can-be-used-either-with-or-without-paramet
    '''
    @functools.wraps(f)
    def new_dec(*args, **kwargs):
        if len(args) == 1 and len(kwargs) == 0 and callable(args[0]):
     

In [4]:
# Let's see some statistics about our dataset
code_lengths = [len(item['code']) for item in data]
function_names = [item['name'] for item in data]

# Create a simple summary
df = pd.DataFrame({
    'code_length': code_lengths,
    'has_docstring': [len(item['docstring']) > 0 for item in data]
})

print("Dataset Statistics:")
print("=" * 50)
print(df.describe())
print("\nDocstring coverage:")
print(df['has_docstring'].value_counts())

Dataset Statistics:
        code_length
count  10000.000000
mean     994.745800
std     1299.941164
min      100.000000
25%      372.000000
50%      622.000000
75%     1129.000000
max    24543.000000

Docstring coverage:
has_docstring
True    10000
Name: count, dtype: int64


In [5]:
# Let's find potential code smells using simple rules
def check_basic_smells(code):
    """Check for obvious code smells using simple heuristics"""
    smells = []
    
    # 1. Long function (>50 lines)
    lines = code.split('\n')
    if len(lines) > 50:
        smells.append('long_function')
    
    # 2. Deep nesting (count leading spaces)
    max_indent = max([len(line) - len(line.lstrip()) for line in lines if line.strip()])
    if max_indent > 16:  # More than 4 levels of nesting
        smells.append('deep_nesting')
    
    # 3. Magic numbers (simple check - look for bare numbers)
    import re
    numbers = re.findall(r'\b\d+\b', code)
    if len(numbers) > 10:  # Many magic numbers
        smells.append('magic_numbers')
    
    return smells

# Apply to all samples
for item in data:
    item['auto_detected_smells'] = check_basic_smells(item['code'])

# Count how many have potential smells
smell_counts = {}
for item in data:
    for smell in item['auto_detected_smells']:
        smell_counts[smell] = smell_counts.get(smell, 0) + 1

print("Potential smells found:")
for smell, count in smell_counts.items():
    print(f"  {smell}: {count} functions")

Potential smells found:
  deep_nesting: 2629 functions
  magic_numbers: 912 functions
  long_function: 946 functions


In [7]:
# Let's examine some examples with potential smells

# Find one example of each smell type
examples = {
    'deep_nesting': [],
    'long_function': [],
    'magic_numbers': []
}

for item in data:
    for smell in item['auto_detected_smells']:
        if len(examples[smell]) < 3:  # Get 3 examples of each
            examples[smell].append(item)

# Show first deep nesting example
print("EXAMPLE: Potential Deep Nesting")
print("="*60)
print(f"Function: {examples['deep_nesting'][0]['name']}")
print("="*60)
print(examples['deep_nesting'][0]['code'])
# print("..." if len(examples['deep_nesting'][0]['code']) > 800 else "")

EXAMPLE: Potential Deep Nesting
Function: ImageGraphCut.__msgc_step3_discontinuity_localization
def __msgc_step3_discontinuity_localization(self):
        """
        Estimate discontinuity in basis of low resolution image segmentation.
        :return: discontinuity in low resolution
        """
        import scipy

        start = self._start_time
        seg = 1 - self.segmentation.astype(np.int8)
        self.stats["low level object voxels"] = np.sum(seg)
        self.stats["low level image voxels"] = np.prod(seg.shape)
        # in seg is now stored low resolution segmentation
        # back to normal parameters
        # step 2: discontinuity localization
        # self.segparams = sparams_hi
        seg_border = scipy.ndimage.filters.laplace(seg, mode="constant")
        logger.debug("seg_border: %s", scipy.stats.describe(seg_border, axis=None))
        # logger.debug(str(np.max(seg_border)))
        # logger.debug(str(np.min(seg_border)))
        seg_border[seg_border != 0] = 

In [8]:
print("\nEXAMPLE: Potential Long Function")
print("="*60)
print(f"Function: {examples['long_function'][0]['name']}")
print(f"Lines of code: {len(examples['long_function'][0]['code'].split(chr(10)))}")
print("="*60)
print(examples['long_function'][0]['code'])


EXAMPLE: Potential Long Function
Function: ImageGraphCut.__multiscale_gc_lo2hi_run
Lines of code: 81
def __multiscale_gc_lo2hi_run(self):  # , pyed):
        """
        Run Graph-Cut segmentation with refinement of low resolution multiscale graph.
        In first step is performed normal GC on low resolution data
        Second step construct finer grid on edges of segmentation from first
        step.
        There is no option for use without `use_boundary_penalties`
        """
        # from PyQt4.QtCore import pyqtRemoveInputHook
        # pyqtRemoveInputHook()
        self._msgc_lo2hi_resize_init()
        self.__msgc_step0_init()

        hard_constraints = self.__msgc_step12_low_resolution_segmentation()
        # ===== high resolution data processing
        seg = self.__msgc_step3_discontinuity_localization()

        self.stats["t3.1"] = (time.time() - self._start_time)
        graph = Graph(
            seg,
            voxelsize=self.voxelsize,
            nsplit=self.s

In [9]:
print("\nEXAMPLE: Potential Magic Numbers")
print("="*60)
print(f"Function: {examples['magic_numbers'][0]['name']}")
print("="*60)
print(examples['magic_numbers'][0]['code'])


EXAMPLE: Potential Magic Numbers
Function: ImageGraphCut.__msgc_step3_discontinuity_localization
def __msgc_step3_discontinuity_localization(self):
        """
        Estimate discontinuity in basis of low resolution image segmentation.
        :return: discontinuity in low resolution
        """
        import scipy

        start = self._start_time
        seg = 1 - self.segmentation.astype(np.int8)
        self.stats["low level object voxels"] = np.sum(seg)
        self.stats["low level image voxels"] = np.prod(seg.shape)
        # in seg is now stored low resolution segmentation
        # back to normal parameters
        # step 2: discontinuity localization
        # self.segparams = sparams_hi
        seg_border = scipy.ndimage.filters.laplace(seg, mode="constant")
        logger.debug("seg_border: %s", scipy.stats.describe(seg_border, axis=None))
        # logger.debug(str(np.max(seg_border)))
        # logger.debug(str(np.min(seg_border)))
        seg_border[seg_border != 0] 

In [10]:
# Prepare features and labels for ML

def extract_features(code):
    """Extract numerical features from code"""
    lines = code.split('\n')
    
    features = {}
    
    # Feature 1: Number of lines
    features['num_lines'] = len(lines)
    
    # Feature 2: Average line length
    non_empty_lines = [line for line in lines if line.strip()]
    features['avg_line_length'] = sum(len(line) for line in non_empty_lines) / max(len(non_empty_lines), 1)
    
    # Feature 3: Max indentation (indicator of nesting)
    features['max_indent'] = max([len(line) - len(line.lstrip()) for line in lines if line.strip()], default=0)
    
    # Feature 4: Number of numeric literals (magic numbers)
    import re
    numbers = re.findall(r'\b\d+\b', code)
    features['num_literals'] = len(numbers)
    
    # Feature 5: Number of comments
    features['num_comments'] = sum(1 for line in lines if '#' in line)
    
    return features

# Test it on one example
sample_code = data[0]['code']
sample_features = extract_features(sample_code)

print("Example features extracted from code:")
print(sample_features)

Example features extracted from code:
{'num_lines': 50, 'avg_line_length': 43.4468085106383, 'max_indent': 20, 'num_literals': 11, 'num_comments': 12}


In [11]:
# Extract features for entire dataset
print("Extracting features from all 10,000 samples...")
print("This might take a minute...\n")

import pandas as pd

# Extract features for each sample
all_features = []
all_labels = []

for item in data:
    features = extract_features(item['code'])
    all_features.append(features)
    
    # Create label: 1 if has ANY smell, 0 if clean
    has_smell = 1 if len(item['auto_detected_smells']) > 0 else 0
    all_labels.append(has_smell)

# Convert to DataFrame (easier to work with)
features_df = pd.DataFrame(all_features)
features_df['has_smell'] = all_labels

print("✅ Feature extraction complete!")
print(f"\nDataset shape: {features_df.shape}")
print(f"\nFirst few rows:")
print(features_df.head())
print(f"\nLabel distribution:")
print(features_df['has_smell'].value_counts())

Extracting features from all 10,000 samples...
This might take a minute...

✅ Feature extraction complete!

Dataset shape: (10000, 6)

First few rows:
   num_lines  avg_line_length  max_indent  num_literals  num_comments  \
0         50        43.446809          20            11            12   
1         81        42.847222          12            16            17   
2         19        49.333333          12             0             4   
3         51        38.025000          16            28            17   
4         42        46.323529          18            16            13   

   has_smell  
0          1  
1          1  
2          0  
3          1  
4          1  

Label distribution:
has_smell
0    6773
1    3227
Name: count, dtype: int64


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Separate features (X) and labels (y)
X = features_df.drop('has_smell', axis=1)  # All columns except label
y = features_df['has_smell']  # Just the label

# Split into train (80%) and test (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

Training samples: 8000
Testing samples: 2000


In [15]:
print("\nTraining the model...")
# Train a Random Forest classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("✅ Training complete!")

# Make predictions on test set
y_pred = model.predict(X_test)

# Evaluate
print("\n" + "="*60)
print("MODEL PERFORMANCE")
print("="*60)
print(classification_report(y_test, y_pred, target_names=['Clean', 'Has Smell']))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Training the model...
✅ Training complete!

MODEL PERFORMANCE
              precision    recall  f1-score   support

       Clean       1.00      1.00      1.00      1330
   Has Smell       1.00      1.00      1.00       670

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000


Confusion Matrix:
[[1330    0]
 [   0  670]]


In [17]:
from transformers import AutoTokenizer, AutoModel
import torch

print("Loading CodeBERT model and tokenizer...")
print("This will download ~500MB on first run (one-time download)\n")

# Load pre-trained CodeBERT
model_name = "microsoft/codebert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("✅ CodeBERT loaded successfully!")
print(f"Vocabulary size: {tokenizer.vocab_size}")

Loading CodeBERT model and tokenizer...
This will download ~500MB on first run (one-time download)

✅ CodeBERT loaded successfully!
Vocabulary size: 50265


In [18]:
# Take a sample code snippet
sample_code = data[0]['code']

print("Original code (first 200 chars):")
print("="*60)
print(sample_code[:200])
print("\n" + "="*60)

# Tokenize it
tokens = tokenizer.encode(sample_code, truncation=True, max_length=512)

print(f"\nTokenized into {len(tokens)} tokens")
print(f"First 20 tokens (as numbers): {tokens[:20]}")

# Decode back to see what tokens look like
print("\nFirst 20 tokens (as text):")
decoded = tokenizer.convert_ids_to_tokens(tokens[:20])
print(decoded)

Original code (first 200 chars):
def __msgc_step3_discontinuity_localization(self):
        """
        Estimate discontinuity in basis of low resolution image segmentation.
        :return: discontinuity in low resolution
        ""


Tokenized into 512 tokens
First 20 tokens (as numbers): [0, 9232, 27148, 48593, 438, 1215, 13975, 246, 1215, 18909, 2533, 179, 15147, 1215, 18076, 1938, 1640, 13367, 3256, 50118]

First 20 tokens (as text):
['<s>', 'def', 'Ġ__', 'msg', 'c', '_', 'step', '3', '_', 'disc', 'ont', 'in', 'uity', '_', 'local', 'ization', '(', 'self', '):', 'Ċ']


In [19]:
print("Tokenizing all 10,000 code samples...")
print("This will take 1-2 minutes...\n")

# Tokenize all code samples
all_tokenized = []
all_labels = []

for item in data:
    # Tokenize the code
    tokens = tokenizer.encode(
        item['code'],
        truncation=True,
        max_length=512,
        padding='max_length',  # Pad shorter sequences to 512
        return_tensors='pt'    # Return PyTorch tensors
    )
    
    all_tokenized.append(tokens.squeeze())  # Remove extra dimension
    
    # Label: 1 if has smell, 0 if clean
    label = 1 if len(item['auto_detected_smells']) > 0 else 0
    all_labels.append(label)

# Convert to tensors
X_tokens = torch.stack(all_tokenized)
y_labels = torch.tensor(all_labels)

print(f"✅ Tokenization complete!")
print(f"Input shape: {X_tokens.shape}")  # Should be (10000, 512)
print(f"Labels shape: {y_labels.shape}")  # Should be (10000,)
print(f"\nLabel distribution:")
print(f"  Clean (0): {(y_labels == 0).sum().item()}")
print(f"  Has smell (1): {(y_labels == 1).sum().item()}")

Tokenizing all 10,000 code samples...
This will take 1-2 minutes...

✅ Tokenization complete!
Input shape: torch.Size([10000, 512])
Labels shape: torch.Size([10000])

Label distribution:
  Clean (0): 6773
  Has smell (1): 3227


In [20]:
from sklearn.model_selection import train_test_split

# Split into train (80%) and test (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X_tokens, y_labels, test_size=0.2, random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

Training samples: 8000
Testing samples: 2000


In [21]:
import torch.nn as nn

class CodeSmellClassifier(nn.Module):
    def __init__(self):
        super(CodeSmellClassifier, self).__init__()
        
        # Load CodeBERT
        self.codebert = AutoModel.from_pretrained("microsoft/codebert-base")
        
        # Add a classifier on top (768 is CodeBERT's output size)
        self.classifier = nn.Linear(768, 2)  # 2 classes: clean or smell
        
    def forward(self, input_ids):
        # Get CodeBERT embeddings
        outputs = self.codebert(input_ids)
        
        # Use the [CLS] token representation (first token)
        cls_output = outputs.last_hidden_state[:, 0, :]
        
        # Classify
        logits = self.classifier(cls_output)
        return logits

# Create the model
print("Building the classifier...")
classifier = CodeSmellClassifier()
print("✅ Model built!")
print(f"Total parameters: {sum(p.numel() for p in classifier.parameters()):,}")

Building the classifier...


pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Model built!
Total parameters: 124,647,170


In [22]:
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim

# Create datasets
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

# Create data loaders (batch processing)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

# Setup training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

classifier = classifier.to(device)
optimizer = optim.Adam(classifier.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

print("\n✅ Training setup complete!")
print(f"Batches per epoch: {len(train_loader)}")

Using device: cpu

✅ Training setup complete!
Batches per epoch: 1000
